# Day 1, Movement 03. Stance demo on real US trade reports

**Instructor notebook.** Runs on paid Colab Pro with GPU.

We load 1,432 IP-section paragraphs from US National Trade Estimate reports (1995 to 2022), point an off-the-shelf zero-shot DeBERTa model at them, and compare what the off-the-shelf model produces against the fine-tuned scores from Park's paper. The point of Day 1 is to see what off-the-shelf can do. Day 2 is where we close the gap with fine-tuning and validation.

**Before you run.** Make sure `NTE_IPR_final_v2.csv` is uploaded to your Google Drive at `MyDrive/nte_text/NTE_IPR_final_v2.csv`.

## 1. Setup

Mount Drive, install `transformers`, set up imports.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers

In [ ]:
import time
import pandas as pd
import torch
from transformers import pipeline

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. Load the NTE corpus

Park's replication corpus, 1995 to 2022. Four columns. `country`, `year`, `text` (the IP-section paragraph), and `deberta_score` (her fine-tuned model's score, scaled to -5 to +5).

For participants who want to replicate after the workshop, the data lives in the `nteText` R package at `github.com/jacqpark/nteText`. Today we read the raw CSV directly.

In [ ]:
CSV_PATH = '/content/drive/MyDrive/nte_text/NTE_IPR_final_v2.csv'
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')
print(f'Years: {df.year.min()} to {df.year.max()}')
print(f'Countries: {df.country.nunique()}')
print(f'Score: {df.deberta_score.min():.2f} to {df.deberta_score.max():.2f}')
df.head(3)

## 3. Pick three paragraphs to show on screen

One clearly negative, one clearly positive, one mixed. Park's fine-tuned scores will be the ground truth we compare against.

In [ ]:
df['len'] = df['text'].str.len()
showcase = df[(df['len'] > 250) & (df['len'] < 700)].copy()

neg = showcase.nsmallest(1, 'deberta_score').iloc[0]
pos = showcase.nlargest(1, 'deberta_score').iloc[0]
mid = showcase[showcase['deberta_score'].between(-0.3, 0.3)].sample(1, random_state=42).iloc[0]

for label, row in [('NEGATIVE', neg), ('POSITIVE', pos), ('MIXED', mid)]:
    print(f'=== {label} | {row.country} {row.year} | fine-tuned score = {row.deberta_score:+.2f} ===')
    print(row.text[:400] + ('...' if len(row.text) > 400 else ''))
    print()

## 4. Load the off-the-shelf zero-shot encoder

`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`. ~184M params, MIT license. Trained on three NLI corpora. We will use it through the `zero-shot-classification` pipeline with our own candidate labels.

In [ ]:
MODEL_ID = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
device = 0 if torch.cuda.is_available() else -1

zsc = pipeline('zero-shot-classification', model=MODEL_ID, device=device)
print(f'Loaded {MODEL_ID}')
print(f'Running on: {"GPU" if device == 0 else "CPU"}')

## 5. Run zero-shot on the three showcase paragraphs

Three candidate labels that mirror Park's stance dimension. We wrap each paragraph in a country-year-aware premise so the model has institutional context. This format follows Park's appendix.

In [ ]:
def make_premise(country, year, body):
    return (f'This text is about the IPR protection situation in country '
            f'{country} and year {int(year)}: {body}')

candidate_labels = [
    'the country has weak IP protection and US criticizes it',
    'the country has strong IP protection and US praises it',
    'unrelated to IP protection',
]

for label, row in [('NEGATIVE', neg), ('POSITIVE', pos), ('MIXED', mid)]:
    premise = make_premise(row.country, row.year, row.text)
    out = zsc(premise, candidate_labels=candidate_labels)
    print(f'--- {label} | {row.country} {row.year} | fine-tuned={row.deberta_score:+.2f} ---')
    for lbl, sc in zip(out['labels'], out['scores']):
        print(f'  {sc:0.3f}  {lbl}')
    print()

## 6. Run zero-shot on a 200-paragraph sample and compare to Park's fine-tuned score

We compute a simple zero-shot stance score: probability of "weak IP" minus probability of "strong IP". Higher means the encoder thinks the paragraph is more critical of IP protection.

Then we correlate against Park's fine-tuned `deberta_score`. If zero-shot does anything sensible, the correlation will be negative (high zero-shot critical score should track low fine-tuned score).

In [ ]:
sample = df.sample(200, random_state=0).reset_index(drop=True)
sample['premise'] = [make_premise(c, y, t) for c, y, t in zip(sample['country'], sample['year'], sample['text'])]

t0 = time.time()
preds = zsc(sample['premise'].tolist(), candidate_labels=candidate_labels, batch_size=8)
elapsed = time.time() - t0

def critic_score(p):
    d = dict(zip(p['labels'], p['scores']))
    return d['the country has weak IP protection and US criticizes it'] - d['the country has strong IP protection and US praises it']

sample['zsc_critic'] = [critic_score(p) for p in preds]
print(f'Inference: {elapsed:.1f} sec for 200 paragraphs ({elapsed/200*1000:.0f} ms each)')
print(f'Pearson(zsc_critic, fine-tuned deberta_score) = {sample[["zsc_critic","deberta_score"]].corr().iloc[0,1]:+.3f}')
print('A negative correlation means zero-shot tracks fine-tuned in the right direction.')

## 7. Show the misses

The headline correlation hides the variance. Day 2 will dig into why interpretive paragraphs trip zero-shot models, and how Park's 13-hypothesis ensemble (Table A.1 of the appendix) closes the gap.

In [ ]:
sample['gap'] = sample['zsc_critic'] - (-sample['deberta_score'] / 5)
worst = sample.reindex(sample['gap'].abs().sort_values(ascending=False).index).head(5)
for _, r in worst.iterrows():
    print(f'[{r.country} {r.year}] fine-tuned={r.deberta_score:+.2f}  zsc_critic={r.zsc_critic:+.2f}')
    print('  ' + r.text[:280] + '...')
    print()

## 8. CPU vs GPU timing

The same 100 paragraphs, same model, two devices. This is the slide that justifies why Day 1 spends 30 minutes on Colab.

In [ ]:
timing_sample = df.sample(100, random_state=1)['text'].tolist()

if torch.cuda.is_available():
    zsc_gpu = pipeline('zero-shot-classification', model=MODEL_ID, device=0)
    t0 = time.time()
    _ = zsc_gpu(timing_sample, candidate_labels=candidate_labels, batch_size=8)
    gpu_time = time.time() - t0
    print(f'GPU: {gpu_time:.1f} sec for 100 paragraphs')
    del zsc_gpu
    torch.cuda.empty_cache()

zsc_cpu = pipeline('zero-shot-classification', model=MODEL_ID, device=-1)
t0 = time.time()
_ = zsc_cpu(timing_sample, candidate_labels=candidate_labels, batch_size=8)
cpu_time = time.time() - t0
print(f'CPU: {cpu_time:.1f} sec for 100 paragraphs')

if torch.cuda.is_available():
    print(f'Speedup: {cpu_time/gpu_time:.1f}x')

## What we just saw

1. Zero-shot DeBERTa on country-year-aware NTE premises with 3 candidate labels gives a stance score that correlates in the right direction with Park's fine-tuned score.
2. The headline correlation hides real misses, especially on interpretive paragraphs that mix praise and criticism.
3. GPU is roughly 10x to 30x faster than CPU on this workload. Free Colab CPU is fine for thousands of paragraphs. For tens of thousands, you need GPU.

**Day 2 picks up here.** We replace the three labels with Park's 13 weighted hypotheses from Table A.1, hand-code a 30-paragraph gold set, validate against it, and turn predictions into a defensible measurement proxy.